# Projeto Memosyne: Reconhecimento de Peças de Xadrez

> ⚠️ **ATENÇÃO: Base de Dados (PGN e FEN)**
> Para quem for testar este notebook, recomendamos baixar o seu próprio arquivo de partidas de xadrez (.pgn) em bases públicas (como Lichess ou Kaggle) e gerar os seus próprios .txt de FENs e imagens .png. Os arquivos inclusos no repositório são apenas para fins de demonstração.


## Importações

In [ ]:
from __future__ import annotations

import gc
import os
import random
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple, cast

import cairosvg
import chess
import chess.svg
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras # type: ignore


In [ ]:
# =====================================================================
# CONFIGURAÇÕES PADRÃO UTILIZADAS NO NOTEBOOK ORIGINAL
# =====================================================================
ARQUIVO_FENS = Path("fens_extraidos.txt")
PASTA_DATASET = Path("dataset_final")
LIMITE_FENS_PARA_PROCESSAR = 4_000
TAMANHO_IMAGEM_SAIDA = 64
MARGEM_FATIAMENTO = 2
IMG_SIZE: Tuple[int, int] = (64, 64)
BATCH_SIZE = 32
EPOCHS = 20
PREFETCH_SIZE = 2
VALIDATION_SPLIT = 0.2
SEED = 123
TAMANHO_PADRAO_RECONHECIMENTO = 800
MARGEM_RECONHECIMENTO = 7
IMAGEM_RECONHECIMENTO = Path("tabuleiro_teste.jpg")
CORES_POSSIVEIS: Sequence[Dict[str, str]] = [
    {"square light": "#f2d3b1", "square dark": "#a97b65"},
    {"square light": "#e8ebf0", "square dark": "#7d8796"},
    {"square light": "#ffffff", "square dark": "#404040"},
    {"square light": "#baca44", "square dark": "#718339"},
    {"square light": "#f7f7f7", "square dark": "#8c8c8c"},
]
PECAS_MAP = {
    "P": "wP",
    "p": "bP",
    "N": "wN",
    "n": "bN",
    "B": "wB",
    "b": "bB",
    "R": "wR",
    "r": "bR",
    "Q": "wQ",
    "q": "bQ",
    "K": "wK",
    "k": "bK",
}
REV_PECAS_MAP = {valor: chave for chave, valor in PECAS_MAP.items()}
REV_PECAS_MAP["empty"] = "1"


### ROTINA DE CRIAÇÃO DO DATASET


In [8]:
# =====================================================================
# ROTINA DE CRIAÇÃO DO DATASET
# =====================================================================
def _validar_bytes(png_bytes: bytes | None) -> bytes:
    if png_bytes is None:
        raise ValueError("cairosvg.svg2png retornou None; verifique o SVG gerado.")
    return png_bytes


def _validar_imagem(imagem_tabuleiro: np.ndarray | None) -> np.ndarray:
    if imagem_tabuleiro is None:
        raise ValueError("cv2.imdecode falhou ao reconstruir a imagem do tabuleiro.")
    return imagem_tabuleiro


def gerar_dataset(
    arquivo_fens: Path = ARQUIVO_FENS,
    pasta_dataset: Path = PASTA_DATASET,
    limite_fens: int = LIMITE_FENS_PARA_PROCESSAR,
    tamanho_saida: int = TAMANHO_IMAGEM_SAIDA,
    margem: int = MARGEM_FATIAMENTO,
    cores_possiveis: Sequence[Dict[str, str]] = CORES_POSSIVEIS,
) -> None:
    pasta_dataset.mkdir(parents=True, exist_ok=True)
    nomes_pastas = list(PECAS_MAP.values()) + ["empty"]

    for nome in nomes_pastas:
        (pasta_dataset / nome).mkdir(parents=True, exist_ok=True)

    with arquivo_fens.open("r", encoding="utf-8") as ponteiro:
        fens = ponteiro.readlines()

    contadores_pecas = {nome: 0 for nome in nomes_pastas}

    for indice, fen in enumerate(fens[:limite_fens]):
        fen_limpo = fen.strip()
        try:
            board = chess.Board(fen_limpo)
        except ValueError as exc:
            print(f"FEN inválido na linha {indice + 1}: {fen_limpo} ({exc})")
            continue

        cores = random.choice(cores_possiveis)
        svg_code = chess.svg.board(board=board, size=400, colors=cores)
        svg_bytes = svg_code.encode("utf-8")
        png_bytes = _validar_bytes(cairosvg.svg2png(bytestring=svg_bytes))

        np_array = np.frombuffer(png_bytes, dtype=np.uint8)
        imagem_tabuleiro = _validar_imagem(cv2.imdecode(np_array, cv2.IMREAD_COLOR))

        altura, largura = imagem_tabuleiro.shape[:2]
        tamanho_casa = largura // 8

        for linha in range(8):
            for coluna in range(8):
                y1 = linha * tamanho_casa + margem
                y2 = (linha + 1) * tamanho_casa - margem
                x1 = coluna * tamanho_casa + margem
                x2 = (coluna + 1) * tamanho_casa - margem

                img_casa = imagem_tabuleiro[y1:y2, x1:x2]
                img_casa_redimensionada = cv2.resize(
                    img_casa, (tamanho_saida, tamanho_saida), interpolation=cv2.INTER_AREA
                )

                quadrado = chess.square(coluna, 7 - linha)
                peca = board.piece_at(quadrado)
                if peca:
                    nome_pasta = PECAS_MAP.get(peca.symbol())
                    if nome_pasta is None:
                        raise KeyError(f"Peça inesperada encontrada: {peca.symbol()}")
                else:
                    nome_pasta = "empty"

                contadores_pecas[nome_pasta] += 1
                nome_arquivo = f"{contadores_pecas[nome_pasta]:04d}.png"
                caminho_final = pasta_dataset / nome_pasta / nome_arquivo
                cv2.imwrite(str(caminho_final), img_casa_redimensionada)

        if (indice + 1) % 50 == 0:
            print(f"Processados {indice + 1} FENs.")


### PREPARAÇÃO DO DATASET PARA TREINAMENTO

In [10]:
# =====================================================================
# PREPARAÇÃO DO DATASET PARA TREINAMENTO
# =====================================================================
def preparar_datasets(
    dataset_dir: Path,
    img_size: Tuple[int, int],
    batch_size: int,
    validation_split: float,
    seed: int,
    prefetch_size: int,
) -> Tuple[tf.data.Dataset, tf.data.Dataset, List[str]]:
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Diretório do dataset não encontrado: {dataset_dir}")

    class_names = sorted(
        [
            nome
            for nome in os.listdir(dataset_dir)
            if (dataset_dir / nome).is_dir()
        ]
    )
    if not class_names:
        raise ValueError("Nenhuma classe encontrada no diretório do dataset.")

    dataset_args = dict(
        validation_split=validation_split,
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
    )

    train_ds = cast(
        tf.data.Dataset,
        keras.utils.image_dataset_from_directory(
            dataset_dir, subset="training", **dataset_args
        ),
    )
    val_ds = cast(
        tf.data.Dataset,
        keras.utils.image_dataset_from_directory(
            dataset_dir, subset="validation", **dataset_args
        ),
    )

    train_ds = train_ds.prefetch(prefetch_size)
    val_ds = val_ds.prefetch(prefetch_size)

    return train_ds, val_ds, class_names


### CONSTRUÇÃO E TREINAMENTO DO MODELO

In [ ]:
# =====================================================================
# CONSTRUÇÃO E TREINAMENTO DO MODELO
# =====================================================================
def construir_modelo(num_classes: int) -> keras.Sequential:
    model = keras.Sequential(
        [
            keras.layers.Rescaling(1.0 / 255),
            keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Flatten(),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(num_classes, activation="softmax"),
        ]
    )

    optimizer: tf.keras.optimizers.Optimizer = tf.keras.optimizers.Adam( # type: ignore
        learning_rate=0.001
    )
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def treinar_modelo(
    model: keras.Model,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    epochs: int,
    checkpoint_path: Path = Path("melhor_modelo_checkpoint.keras"),
) -> keras.callbacks.History:
    checkpoint_callback = keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    )
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    )

    inicio = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[checkpoint_callback, early_stopping, reduce_lr],
        verbose="auto",
    )
    fim = time.time()

    print(
        f"Treinamento concluído em {(fim - inicio)/60:.2f} minutos. "
        f"Melhor acurácia: {max(history.history['val_accuracy']):.4f}. "
        f"Melhor loss: {min(history.history['val_loss']):.4f}."
    )
    return history


### RECONHECIMENTO A PARTIR DE UMA IMAGEM

In [ ]:
# =====================================================================
# RECONHECIMENTO A PARTIR DE UMA IMAGEM
# =====================================================================
def reconhecer_posicao(
    model: keras.Model,
    imagem_path: Path,
    class_names: Sequence[str],
    img_size: Tuple[int, int] = IMG_SIZE,
    tamanho_tabuleiro: int = TAMANHO_PADRAO_RECONHECIMENTO,
    margem: int = MARGEM_RECONHECIMENTO,
) -> str:
    imagem = cv2.imread(str(imagem_path))
    if imagem is None:
        raise FileNotFoundError(f"Imagem para reconhecimento não encontrada: {imagem_path}")

    tabuleiro = cv2.resize(imagem, (tamanho_tabuleiro, tamanho_tabuleiro))
    tamanho_casa = tabuleiro.shape[0] // 8
    linhas_fen = []

    for i in range(8):
        linha = ""
        casas_vazias = 0
        for j in range(8):
            y1 = i * tamanho_casa + margem
            y2 = (i + 1) * tamanho_casa - margem
            x1 = j * tamanho_casa + margem
            x2 = (j + 1) * tamanho_casa - margem
            img_casa = tabuleiro[y1:y2, x1:x2]
            img_casa = cv2.resize(img_casa, img_size)
            img_casa = img_casa.astype("float32")
            img_casa = np.expand_dims(img_casa, axis=0)

            pred = model.predict(img_casa, verbose=0)
            idx = int(np.argmax(pred))
            nome_classe = class_names[idx]
            caractere = REV_PECAS_MAP.get(nome_classe, "1")

            if caractere == "1":
                casas_vazias += 1
            else:
                if casas_vazias > 0:
                    linha += str(casas_vazias)
                    casas_vazias = 0
                linha += caractere

        if casas_vazias > 0:
            linha += str(casas_vazias)
        linhas_fen.append(linha)

    return "/".join(linhas_fen)


### EXECUÇÃO PADRÃO

In [16]:
# =====================================================================
# EXECUÇÃO PADRÃO
# =====================================================================
def executar_pipeline() -> None:
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("GPU configurada para crescimento dinâmico de memória.")
        except RuntimeError as exc:
            print(f"Aviso GPU: {exc}")
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
    gerar_dataset()
    train_ds, val_ds, class_names = preparar_datasets(
        dataset_dir=PASTA_DATASET,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        seed=SEED,
        prefetch_size=PREFETCH_SIZE,
    )
    model = construir_modelo(num_classes=len(class_names))
    history = treinar_modelo(model, train_ds, val_ds, epochs=EPOCHS)
    model.save("meu_modelo_de_xadrez_v3.keras")
    if IMAGEM_RECONHECIMENTO.exists():
        try:
            fen_resultado = reconhecer_posicao(model, IMAGEM_RECONHECIMENTO, class_names)
            print(f"FEN reconhecida na imagem '{IMAGEM_RECONHECIMENTO}': {fen_resultado}")
        except Exception as exc:
            print(f"Aviso: falha ao reconhecer a imagem '{IMAGEM_RECONHECIMENTO}': {exc}")
    else:
        print(f"Aviso: imagem '{IMAGEM_RECONHECIMENTO}' não encontrada; reconhecimento omitido.")
    gc.collect()
    keras.backend.clear_session()
    print("Execução concluída. Modelo salvo em 'meu_modelo_de_xadrez_v3.keras'.")
    print(f"Histórico disponível com as chaves: {list(history.history.keys())}")


In [ ]:
# Executar o pipeline completo
executar_pipeline()


GPU configurada para crescimento dinâmico de memória.
Processados 50 FENs.
Processados 100 FENs.
Processados 150 FENs.
Processados 200 FENs.
Processados 250 FENs.
Processados 300 FENs.
Processados 350 FENs.
Processados 400 FENs.
Processados 450 FENs.
Processados 500 FENs.
Processados 550 FENs.
Processados 600 FENs.
Processados 650 FENs.
Processados 700 FENs.
Processados 750 FENs.
Processados 800 FENs.
Processados 850 FENs.
Processados 900 FENs.
Processados 950 FENs.
Processados 1000 FENs.
Processados 1050 FENs.
Processados 1100 FENs.
Processados 1150 FENs.
Processados 1200 FENs.
Processados 1250 FENs.
Processados 1300 FENs.
Processados 1350 FENs.
Processados 1400 FENs.
Processados 1450 FENs.
Processados 1500 FENs.
Processados 1550 FENs.
Processados 1600 FENs.
Processados 1650 FENs.
Processados 1700 FENs.
Processados 1750 FENs.
Processados 1800 FENs.
Processados 1850 FENs.
Processados 1900 FENs.
Processados 1950 FENs.
Processados 2000 FENs.
Processados 2050 FENs.
Processados 2100 FENs.


NameError: name 'IMAGEM_RECONHECIMENTO' is not defined